# 02 - Data Preprocessing
Load filtered dataset from Week 2, handle missing values, outliers, encoding, and train/test split.

In [ ]:
import pandas as pd
import numpy as np
import glob
import matplotlib.pyplot as plt
import seaborn as sns
import os

## 1. Load and Combine Data 
#### (same as 01_exploration)

In [ ]:
# Find all CRMLS files, exclude the pre-processed one
file_list = glob.glob("data/CRMLSSold*.csv")
clean_files = [f for f in file_list if "_filled" not in f]

# Load and combine all files
df_list = []
for file in clean_files:
    temp_df = pd.read_csv(file, low_memory=False)
    df_list.append(temp_df)

df = pd.concat(df_list, ignore_index=True)

# Apply the dual filter
df = df[(df['PropertyType'] == 'Residential') & (df['PropertySubType'] == 'SingleFamilyResidence')]

print('Shape after load + filter:', df.shape)

## 2. Handle ClosePrice Outliers
Rows with ClosePrice > $100M are data entry errors, not real sales. 

Drop these before imputing other columns so the median isn't skewed.

In [ ]:
print('Rows with ClosePrice > $100M:', (df['ClosePrice'] > 100000000).sum())

# Remove unrealistic prices
df = df[df['ClosePrice'] <= 100000000]

# Remove $0 or negative prices — log-transform can't handle these
print('Rows with ClosePrice <= 0:', (df['ClosePrice'] <= 0).sum())
df = df[df['ClosePrice'] > 0]

# Drop the 1 row missing ClosePrice entirely (can't impute a target variable)
df = df.dropna(subset=['ClosePrice'])

print('Shape after cleaning ClosePrice:', df.shape)

## 3. Handle Missing Values in Other Key Columns
Using median fill since it's robust to outliers.

In [ ]:
print('Missing before:')
print(df[['LivingArea', 'BathroomsTotalInteger', 'YearBuilt']].isnull().sum())

df['LivingArea'] = df['LivingArea'].fillna(df['LivingArea'].median())
df['BathroomsTotalInteger'] = df['BathroomsTotalInteger'].fillna(df['BathroomsTotalInteger'].median())
df['YearBuilt'] = df['YearBuilt'].fillna(df['YearBuilt'].median())

print('\nMissing after:')
print(df[['LivingArea', 'BathroomsTotalInteger', 'YearBuilt']].isnull().sum())

## 4. Handle LotSizeSquareFeet (outliers + missing)
Cap unrealistic lot sizes (>10 acres = 435,600 sqft) as missing first, then fill with median.

In [ ]:
lot_cutoff = 435600  # 10 acres in sqft

print('Outlier rows (>10 acres):', (df['LotSizeSquareFeet'] > lot_cutoff).sum())

# Treat extreme outliers as missing
df.loc[df['LotSizeSquareFeet'] > lot_cutoff, 'LotSizeSquareFeet'] = np.nan

print('Missing before fill:', df['LotSizeSquareFeet'].isnull().sum())
df['LotSizeSquareFeet'] = df['LotSizeSquareFeet'].fillna(df['LotSizeSquareFeet'].median())
print('Missing after fill:', df['LotSizeSquareFeet'].isnull().sum())

In [ ]:
# Fill remaining numeric NaNs with median
numeric_cols_with_na = ['FireplacesTotal', 'AboveGradeFinishedArea', 'TaxAnnualAmount',
                         'ParkingTotal', 'LotSizeAcres', 'StreetNumberNumeric', 'TaxYear',
                         'BuildingAreaTotal', 'BelowGradeFinishedArea', 'CoveredSpaces',
                         'Stories', 'LotSizeArea', 'MainLevelBedrooms', 'GarageSpaces',
                         'AssociationFee']

for col in numeric_cols_with_na:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())

print('Remaining NaNs in these cols:', df[numeric_cols_with_na].isna().sum().sum())

In [ ]:
print(df[numeric_cols_with_na].isna().mean().sort_values(ascending=False))

In [ ]:
fully_empty_cols = ['FireplacesTotal', 'AboveGradeFinishedArea', 'TaxAnnualAmount',
                     'TaxYear', 'CoveredSpaces']
df = df.drop(columns=[c for c in fully_empty_cols if c in df.columns])
print('Shape after dropping empty columns:', df.shape)
print('Any NaNs left?', df.isna().sum().sum())

## 5. Identify Rows Missing Lat/Long
These need geocoding (handled separately in 03_geocoding.ipynb).

In [ ]:
missing_geo = df[df['Latitude'].isna() | df['Longitude'].isna()]
print('Rows missing Lat/Long:', len(missing_geo))

missing_geo.to_csv("data/missing_geo_rows.csv", index=False)

In [ ]:
df = df.dropna(subset=['Latitude', 'Longitude'])
print('Shape after dropping missing geo rows:', df.shape)

## 6. Log-Transform ClosePrice
Compresses right skew without dropping any records.

In [ ]:
df['ClosePrice_log'] = np.log(df['ClosePrice'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['ClosePrice'], bins=50)
axes[0].set_title('ClosePrice (raw)')
axes[1].hist(df['ClosePrice_log'], bins=50)
axes[1].set_title('ClosePrice (log-transformed)')
plt.show()

In [ ]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(cat_cols)

## 7. Save Checkpoint (before geocoding merge)

In [ ]:
df.to_csv("data/df_before_geocoding.csv", index=False)
print('Saved: data/df_before_geocoding.csv')

In [ ]:
df['CloseDate'] = pd.to_datetime(df['CloseDate'])
df = df[df['CloseDate'] >= '2023-06-01']
print('Shape after June 2023+ filter:', df.shape)

## Drop ID/free-text/leakage columns

In [ ]:
drop_cols = ['ListAgentEmail', 'ListAgentFirstName', 'ListAgentLastName', 'ListAgentFullName',
             'CoListAgentFirstName', 'CoListAgentLastName', 'BuyerAgentMlsId', 'BuyerAgentFirstName',
             'BuyerAgentLastName', 'CoBuyerAgentFirstName', 'ListOfficeName', 'BuyerOfficeName',
             'CoListOfficeName', 'UnparsedAddress', 'ListingId', 'LotSizeDimensions', 'MlsStatus',
             'PropertyType', 'PropertySubType',
             'ContractStatusChangeDate', 'PurchaseContractDate', 'ListingContractDate',
             'ListPrice', 'OriginalListPrice']  # <- Aidan's leakage warning

drop_cols = [c for c in drop_cols if c in df.columns]
df = df.drop(columns=drop_cols)
print('Dropped', len(drop_cols), 'columns')

In [ ]:
# Step 1: Drop any column that's entirely empty (no fillna can fix these)
empty_cols = df.columns[df.isna().all()].tolist()
df = df.drop(columns=empty_cols)
print('Dropped fully-empty columns:', empty_cols)

# Step 2: Fill remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
obj_cols = df.select_dtypes(include='object').columns

for col in num_cols:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].median())

for col in obj_cols:
    if df[col].isna().any():
        df[col] = df[col].fillna('Missing')

print('Any NaNs left?', df.isna().sum().sum())

## Binary Yes/No columns → 0/1

In [ ]:
binary_cols = ['ViewYN', 'WaterfrontYN', 'BasementYN', 'PoolPrivateYN',
               'AttachedGarageYN', 'FireplaceYN', 'NewConstructionYN']

for col in binary_cols:
    if col in df.columns:
        df[col] = df[col].map({'Y': 1, 'N': 0, True: 1, False: 0}).fillna(0)

In [ ]:
for col in ['City', 'PostalCode', 'ElementarySchool', 'SubdivisionName', 'MLSAreaMajor', 'HighSchoolDistrict']:
    print(col, df[col].nunique() if col in df.columns else 'already encoded')

In [ ]:
df = df.drop(columns=[c for c in ['City', 'PostalCode', 'ElementarySchool',
                                    'SubdivisionName', 'MLSAreaMajor', 'HighSchoolDistrict']
                       if c in df.columns])

## One-hot encode remaining categoricals

In [ ]:
encode_cols = ['BuyerAgentAOR', 'ListAgentAOR', 'Flooring', 'AssociationFeeFrequency',
               'BuyerOfficeAOR', 'CountyOrParish', 'BuilderName',
               'BusinessType', 'StateOrProvince', 'MiddleOrJuniorSchool', 'HighSchool', 'Levels',
               'ElementarySchoolDistrict', 'MiddleOrJuniorSchoolDistrict']

encode_cols = [c for c in encode_cols if c in df.columns]
df = pd.get_dummies(df, columns=encode_cols, dummy_na=False)
print('Shape after encoding:', df.shape)

In [ ]:
print('Total columns:', df.shape[1])
print('Columns containing SubdivisionName:', sum(1 for c in df.columns if 'SubdivisionName' in c))
print('Columns containing PostalCode:', sum(1 for c in df.columns if 'PostalCode' in c))

## Temporal train/test split + save

In [ ]:
df['CloseYearMonth'] = df['CloseDate'].dt.to_period('M')
test_month = df['CloseYearMonth'].max()
print('Test month:', test_month)

train_df = df[df['CloseYearMonth'] < test_month]
test_df = df[df['CloseYearMonth'] == test_month]

print('Train shape:', train_df.shape)
print('Test shape:', test_df.shape)

train_df.to_csv("data/train_final.csv", index=False)
test_df.to_csv("data/test_final.csv", index=False)
print("Saved train/test splits")

# 02 - Data Preprocessing — Summary

**What this notebook does:**
1. Loads all 21 CRMLS files, combines them, applies dual filter (Residential + SingleFamilyResidence)
2. Cleans ClosePrice — removes fake outliers (>$100M) and $0/negative rows
3. Fills missing values (LivingArea, BathroomsTotalInteger, YearBuilt) with median
4. Fixes LotSizeSquareFeet outliers (>10 acres treated as missing), fills with median
5. Flags rows missing Lat/Long, saves them separately for geocoding (→ `03_geocoding.ipynb`)
6. Log-transforms ClosePrice to fix right-skew (new column: `ClosePrice_log`)
7. Filters to June 2023+ (team-standardized start date)
8. Drops ID/free-text/leakage columns (including `ListPrice`, `OriginalListPrice` per Aidan's note)
9. Maps binary Yes/No columns to 0/1
10. One-hot encodes remaining categorical variables (⚠️ high-cardinality cols like City/PostalCode/School fields inflate this to ~20,780 columns — candidate for frequency/target encoding later)
11. Temporal train/test split (most recent month = test) → saves `data/train_final.csv` and `data/test_final.csv`

**Still needed (not in this notebook):**
- Geocode missing Lat/Long → `03_geocoding.ipynb`

**Final row count after cleaning:** 230,316 rows (218,292 train / 12,024 test, test month = 2026-05)